# Benchmark Analysis

This notebook uses the benchmark analysis API from `examples/benchmark_runner.py` to:

1. load a full benchmark session
2. group runs by restart family and select the top `K` groups by minimal `best_loss`
3. plot aggregate loss lines for the selected subset
4. load the best model from each selected restart group and scatter plot samples


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from examples.benchmark_runner import (
    entries_to_frame,
    entry_arrays,
    get_lines,
    import_runtime_dependencies,
    load_entry_model,
    load_experiment_entries,
    restart_group_columns,
)


In [ ]:
session_dir = Path('/tmp/opencode/benchmarks/batch_size_cross_entropy_smoke')
entries = load_experiment_entries(session_dir)

print(f'loaded {len(entries)} entries from {session_dir}')
entries[0].keys()


In [ ]:
def select_top_k_restart_groups(entries, k):
    frame = entries_to_frame(entries)
    group_cols = restart_group_columns(frame)

    if not group_cols:
        summary = frame[['entry_index', 'best_loss']].copy()
        summary['group_rank'] = 0
        return entries, summary

    summary = (
        frame.groupby(group_cols, sort=False, dropna=False, as_index=False)
        .agg(
            min_best_loss=('best_loss', 'min'),
            n_restarts=('entry_index', 'count'),
        )
        .sort_values('min_best_loss', ascending=True, kind='stable')
        .head(k)
        .reset_index(drop=True)
    )
    summary['group_rank'] = np.arange(len(summary))

    selected_frame = frame.merge(summary[group_cols], on=group_cols, how='inner')
    selected_indices = selected_frame['entry_index'].tolist()
    subset_entries = [entries[index] for index in selected_indices]
    return subset_entries, summary


def best_entry_per_restart_group(entries):
    frame = entries_to_frame(entries)
    group_cols = restart_group_columns(frame)

    if not group_cols:
        if not entries:
            return []
        best_index = int(frame['best_loss'].idxmin())
        return [entries[int(frame.loc[best_index, 'entry_index'])]]

    best_row_indices = (
        frame.groupby(group_cols, sort=False, dropna=False)['best_loss']
        .idxmin()
        .tolist()
    )
    entry_indices = frame.loc[best_row_indices, 'entry_index'].tolist()
    return [entries[int(index)] for index in entry_indices]


In [ ]:
top_k = 3
subset_entries, subset_summary = select_top_k_restart_groups(entries, top_k)
best_entries = best_entry_per_restart_group(subset_entries)

print(f'selected {len(subset_entries)} entries across {len(best_entries)} restart groups')
subset_summary


In [ ]:
def plot_line_groups(groups):
    for group in groups:
        fig, ax = plt.subplots(figsize=(9, 5), layout='constrained')

        for line in group['lines']:
            style = line.get('style', {})
            color = style.get('color')
            linestyle = style.get('linestyle', '-')
            linewidth = style.get('linewidth', 1.5)

            if 'loss_mean' in line:
                x = np.arange(len(line['loss_mean']))
                ax.plot(
                    x,
                    line['loss_mean'],
                    label=line['label'],
                    color=color,
                    linestyle=linestyle,
                    linewidth=linewidth,
                )
                if 'loss_lower' in line and 'loss_upper' in line:
                    ax.fill_between(
                        x,
                        line['loss_lower'],
                        line['loss_upper'],
                        color=color,
                        alpha=0.2,
                    )
            else:
                loss = np.asarray(line['loss'], dtype=float)
                x = np.arange(len(loss))
                ax.plot(
                    x,
                    loss,
                    label=line['label'],
                    color=color,
                    linestyle=linestyle,
                    linewidth=linewidth,
                )

        ax.set_title(group['title'])
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Loss')
        ax.set_yscale('log')
        ax.grid()
        ax.legend()
        plt.show()


In [ ]:
style_channels = {
    'colormap': ['Blues', 'Reds', 'Greens', 'Purples'],
    'linestyle': ['-', '--', ':', '-.'],
}

groups = get_lines(subset_entries, style_channels)
plot_line_groups(groups)


In [ ]:
def scatter_best_entries(entries, checkpoint_key='best', n_samples=512):
    if not entries:
        return

    deps = import_runtime_dependencies()
    ncols = min(3, len(entries))
    nrows = int(np.ceil(len(entries) / ncols))
    fig, axs = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(5 * ncols, 4 * nrows),
        layout='constrained',
    )
    axs = np.atleast_1d(axs).ravel()

    for ax, entry in zip(axs, entries, strict=False):
        model = load_entry_model(entry, key=checkpoint_key)
        samples = model.sample(n_samples, deps['nnx'].Rngs(0))

        if samples.shape[-1] >= 2:
            ax.scatter(samples[:, 0], samples[:, 1], s=6, alpha=0.7)
            ax.set_xlabel('x[0]')
            ax.set_ylabel('x[1]')
        else:
            ax.hist(samples[:, 0], bins=40)

        ax.set_title(
            f"{entry['run_id']}\n"
            f"best_loss={entry.get('best_loss', np.nan):.3e}, restart={entry.get('restart_index')}"
        )
        ax.grid()

    for ax in axs[len(entries):]:
        ax.axis('off')

    plt.show()


In [ ]:
scatter_best_entries(best_entries, checkpoint_key='best', n_samples=512)


In [ ]:
entries_to_frame(subset_entries).head()
